## Causal Language Modeling (CLM) - Preprocess, Training and Inference

In this lab, we will explore **Causal Language Modeling (CLM)**, which is a core task in training autoregressive language models like GPT-2. CLM is the process of predicting the next word in a sequence, given the previous words. This type of modeling forms the backbone of text generation tasks, where the model learns to generate coherent text by focusing only on previous tokens in the sequence.

The lab is divided into three major sections:
1. **Preprocessing**: Preparing a dataset and the labels for the CLM task and tokenizing them.
2. **Training**: Fine-tuning a pre-trained language model like GPT-2 on a specific dataset using the CLM task.
3. **Inference**: Evaluating the model’s performance by generating text based on input prompts.

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from datasets import load_dataset
import torch
from transformers import set_seed

set_seed(42)
torch.set_num_threads(4)
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Training backend: {device}")

/private/tmp/llms-lab04-venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Training backend: mps


### 1. Preprocessing

1. **Load the Domain-Specific Dataset**:
   - The first step in fine-tuning GPT-2 is to load a dataset that is specific to the domain of interest. In this case, we are using a publicly available **medical dataset** from the **PubMed** collection. PubMed contains a vast number of medical articles, and fine-tuning on such a dataset can help GPT-2 generate more accurate and context-specific medical text.
   - The dataset we're using is `"japhba/pubmed_simple"`, which is a simplified version of PubMed data. This dataset can be easily accessed using the `datasets` library from Hugging Face.


In [2]:
ds = load_dataset("japhba/pubmed_simple", split="train")
train_dataset = ds.shuffle(seed=42).select(range(1000))
eval_dataset = ds.shuffle(seed=42).select(range(1000, 1500))

We can check the contents of any one of the examples in the dataset to understand the structure of the data.

In [3]:
train_dataset[0]

{'abstract': 'Purpose The identification of abnormalities that are relatively rare within otherwise normal anatomy is a major challenge for deep learning in the semantic segmentation of medical images. The small number of samples of the minority classes in the training data makes the learning of optimal classification challenging, while the more frequently occurring samples of the majority class hamper the generalization of the classification boundary between infrequently occurring target objects and classes. In this paper, we developed a novel generative multi-adversarial network, called Ensemble-GAN, for mitigating this class imbalance problem in the semantic segmentation of abdominal images.Method The Ensemble-GAN framework is composed of a single-generator and a multi-discriminator variant for handling the class imbalance problem to provide a better generalization than existing approaches. The ensemble model aggregates the estimates of multiple models by training from different ini

We see that the each entry is a dictionary with two keys:
- "abstract": The abstract of the medical article
- "country": The country where the article was published (we will not use this information in this lab)

2. **Tokenize the Dataset**:
   - The next step is to **tokenize** the data so that it can be processed by GPT-2.
   - GPT-2 does not natively use a padding token, since it does not require fixed length inputs. For this reason, we will substite it with the EOS token as suggested by transformers library (remember, we are also passing an attention mask, so whatever value is used for padding will be ignored by the model!)

In [ ]:
# Tokenize the Dataset using the GPT-2 tokenizer
# Hint: Use the `map` method of the dataset object
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # Set padding token to EOS token

def tokenize_function(samples):
    # Tokenize the text column `abstract` in the dataset (set `truncation=True` and `padding="max_length"`)
    # NOTE: You can either use tokenzier as a global variable, or use a closure to capture the tokenizer variable
    return tokenizer(samples["abstract"], truncation=True, padding="max_length", max_length=1024)

# NOTE: you should map the dataset using the tokenize function. 
# (You can optionally consider using the argument `batched=True` for better performance,
# as long as tokenize_function can handle the batch! [note2: the tokenizer can do that!])
tokenized_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=train_dataset.column_names)
tokenized_dataset_eval = eval_dataset.map(tokenize_function, batched=True, remove_columns=eval_dataset.column_names)

3. **Add labels to the Dataset for Next Token Prediction**:
   - In this step, we will add the **labels** that will be used during the next token prediction task. 
   - In autoregressive language modeling, the **labels** represent the same sequence as the input, used to predict the next token. This is because the model is trained to predict the next token in the sequence given the previous tokens.
   - The shifting of the tokens is already handled internally by the GPT-2 model. Padding labels are set to -100 so they are excluded from the loss. We just pass an extra attribute named `labels` to the dataset (when this argument is passed to the model, it will know to compute the loss for us!)

In [ ]:
# Add labels to the dataset
def add_labels(samples):
    # NOTE: The labels, in causal modeling, should be the same as the input_ids
    samples["labels"] = [
        [token if attended else -100 for token, attended in zip(ids, mask)]
        for ids, mask in zip(samples["input_ids"], samples["attention_mask"])
    ]  # Ignore padding in the loss; GPT-2 shifts labels internally.
    return samples

tokenized_dataset = tokenized_dataset.map(add_labels, batched=True)
tokenized_dataset_eval = tokenized_dataset_eval.map(add_labels, batched=True)

### 2. Training

1. **Fine-Tune the GPT-2 Model**:
   - Set up the model and finetune it using the medical dataset. 
   - The pipeline to be followed is the same that we have already seen in the previous lab (`lab03 - 01-bert`)

In [ ]:
# Set Training Parameters
training_args = TrainingArguments(
    output_dir="./gpt2-medical-finetuned",
    overwrite_output_dir=True,
    num_train_epochs=1,
    # Keep the effective batch size at 6 while fitting a 16 GB Mac.
    per_device_train_batch_size=1,
    gradient_accumulation_steps=6,
    per_device_eval_batch_size=1,
    save_total_limit=2,
    logging_dir="./logs",
    logging_steps=100,
    eval_steps=10,
    eval_strategy="steps",
    report_to="none",
    seed=42,
    dataloader_pin_memory=(device == "cuda"),
)

# Initialize GPT-2 Model
model = AutoModelForCausalLM.from_pretrained(model_name, attn_implementation="sdpa")
model.config.use_cache = False  # KV caches are only needed for generation.

# Fine-Tune the Model with the `Trainer` method and pass also the eval_dataset 
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset_eval,
)

import time
training_start = time.perf_counter()
train_result = trainer.train()
fixed_padding_training_seconds = time.perf_counter() - training_start
print(train_result.metrics)
print(f"Fixed-padding training: {fixed_padding_training_seconds:.2f} seconds")

# Save the Fine-Tuned Model
model.config.use_cache = True
model.save_pretrained("./gpt2-medical-finetuned")
tokenizer.save_pretrained("./gpt2-medical-finetuned")

Step,Training Loss,Validation Loss
10,No log,3.177716
20,No log,3.161057
30,No log,3.160976
40,No log,3.155424
50,No log,3.152704
60,No log,3.151361
70,No log,3.154082
80,No log,3.145792
90,No log,3.144559
100,3.360300,3.138802


{'train_runtime': 2878.7333, 'train_samples_per_second': 0.347, 'train_steps_per_second': 0.058, 'total_flos': 520493727744000.0, 'train_loss': 3.3559414047792733, 'epoch': 0.996}
Fixed-padding training: 2879.03 seconds


('./gpt2-medical-finetuned/tokenizer_config.json',
 './gpt2-medical-finetuned/special_tokens_map.json',
 './gpt2-medical-finetuned/vocab.json',
 './gpt2-medical-finetuned/merges.txt',
 './gpt2-medical-finetuned/added_tokens.json',
 './gpt2-medical-finetuned/tokenizer.json')

### 3. Inference

1. **Compare Text Generation Before and After Fine-Tuning**:
   - Generate text using both the original pre-trained GPT-2 model and the fine-tuned model.
   - Provide the same input prompt and observe the differences in the outputs.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load the pre-trained and fine-tuned GPT-2 models
pretrained_model = AutoModelForCausalLM.from_pretrained(model_name).to(device).eval()
finetuned_model = AutoModelForCausalLM.from_pretrained("./gpt2-medical-finetuned").to(device).eval()

# Tokenize the prompt
# NOTE: fine-tuning does not change the tokenizer, so we can use the same tokenizer for both models)
tokenizer = AutoTokenizer.from_pretrained("./gpt2-medical-finetuned")
tokenizer.pad_token = tokenizer.eos_token

prompt = "The patient presents with chest pain and shortness of breath."

inputs = tokenizer(prompt, return_tensors="pt").to(device)
input_ids = inputs['input_ids']

In [ ]:
# Generate output of the model before fine-tuning (use `generate` and then `decode` methods of the model and tokenizer) 
output_pretrained = pretrained_model.generate(**inputs, max_new_tokens=100, do_sample=False, pad_token_id=tokenizer.eos_token_id)
generated_pretrained = tokenizer.decode(output_pretrained[0], skip_special_tokens=True)

print("**Before fine-tuning (pre-trained GPT-2)**:")
print(generated_pretrained)

**Before Fine-Tuning (Pre-Trained GPT-2)**:
The patient presents with chest pain and shortness of breath. The patient is in a state of shock and is unable to breathe. The patient is in a state of shock and is unable to breathe. The patient is in a state of shock and is unable to breathe. The patient is in a state of shock and is unable to breathe. The patient is in a state of shock and is unable to breathe. The patient is in a state of shock and is unable to breathe. The patient is in a state of shock and is unable to breathe. The patient


In [ ]:
# Generate output of the model after fine-tuning
output_finetuned = finetuned_model.generate(**inputs, max_new_tokens=100, do_sample=False, pad_token_id=tokenizer.eos_token_id)
generated_finetuned = tokenizer.decode(output_finetuned[0], skip_special_tokens=True)

print("**After fine-tuning (fine-tuned on medical dataset)**:")
print(generated_finetuned)

**After Fine-Tuning (Fine-Tuned on Medical Dataset)**:
The patient presents with chest pain and shortness of breath. The patient is in a stable condition and is receiving treatment for a respiratory infection. The patient is in stable condition and is receiving treatment for a respiratory infection. The patient is in stable condition and is receiving treatment for a respiratory infection. The patient is in stable condition and is receiving treatment for a respiratory infection. The patient is in stable condition and is receiving treatment for a respiratory infection. The patient is in stable condition and is receiving treatment for a respiratory infection. The patient is in stable condition and is receiving


<span style="color:red">Extra stuff!</span>

Training the model in this way produces batches with potentially very different lengths. This can be inefficient, as the model will have to pad the sequences to the length of the longest sequence in the batch.

To avoid this, we can use a technique called **Dynamic Padding**. This technique groups the sequences in the batch by length and pads them to the length of the longest sequence in each group. This way, the model only has to pad the sequences to the length of the longest sequence in each group, which can significantly reduce the amount of padding required.

As a first exercise, quantify the number of pad tokens being used in various situations:
1. You pad all batches to the maximum allowed sequence length (1024 for GPT-2, this is what we used so far)
2. You pad the entire batch to the length of the longest sequence in the batch (generate the batches by randomly sampling sentences)
3. You pad the entire batch to the length of the longest sequence in the batch (generate the batches by placing sentences of similar lengths together)

Next, introduce dynamic padding and compare the execution times of the previous execution and the one with dynamic padding.

You can use the following resources to help you with this exercise:
- `group_by_length` parameter ([TrainingArguments](https://huggingface.co/docs/transformers/v4.46.0/en/main_classes/trainer#transformers.TrainingArguments.group_by_length)) (parameter to group together samples with similar lengths)
- `DataCollatorForSeq2Seq` ([DataCollatorForSeq2Seq](https://huggingface.co/docs/transformers/main_classes/data_collator#transformers.DataCollatorForSeq2Seq)) (collator function that aggregates samples into batches and pads them to the maximum length of the batch)

Note: you may find that the validation losses you observe may be different from the previous ones. This is because the cross entropy loss is computed as an average across tokens, and the number of tokens in a batch can vary depending on the padding strategy used.

## Extension: Padding and computational cost

The earlier training cell used fixed-length batches: every abstract became a 1,024-token row. This extension separates two effects: the number of padded token positions created by a batching policy, and the wall-clock time of training with fixed or dynamic padding. The first comparison uses logical groups of six so that grouping by length has a measurable effect. For a controlled, memory-compatible timing check, each training variant starts from the same public GPT-2 checkpoint and processes the same 12 optimizer updates with microbatch one and six-step gradient accumulation. Evaluation is disabled in both short timing runs; this measures training cost, not held-out quality.

Pipeline: tokenized lengths → batch construction → padding counts → padding collator → matched 12-update runs → measured comparison.

In [5]:
# The saved dataset already contains 1,024 positions per row. Recover each
# unpadded prefix using its attention mask; an attended EOS remains content.
import json
import math
import random
import time
from pathlib import Path
import pandas as pd
from torch.utils.data import Dataset

train_lengths = [int(sum(row["attention_mask"])) for row in tokenized_dataset]
eval_lengths = [int(sum(row["attention_mask"])) for row in tokenized_dataset_eval]
assert len(train_lengths) == 1000 and len(eval_lengths) == 500
assert all(0 < length <= 1024 for length in train_lengths + eval_lengths)
print(f"Training examples: {len(train_lengths)}; evaluation examples: {len(eval_lengths)}")
print(f"Training token lengths: min={min(train_lengths)}, median={int(pd.Series(train_lengths).median())}, max={max(train_lengths)}")


Training examples: 1000; evaluation examples: 500
Training token lengths: min=12, median=244, max=823


In [6]:
# Count attended and padded positions for each policy using exactly the same
# training examples. The final group is retained even when it has fewer than 6.
BATCH_SIZE = 6
MAX_LENGTH = 1024

def padded_positions(lengths, batches):
    return sum(len(batch) * max(batch) - sum(batch) for batch in batches)

def grouped_lengths(lengths):
    return [lengths[i:i + BATCH_SIZE] for i in range(0, len(lengths), BATCH_SIZE)]

fixed_padding = len(train_lengths) * MAX_LENGTH - sum(train_lengths)
random_order = list(train_lengths)
random.Random(42).shuffle(random_order)
random_padding = padded_positions(train_lengths, grouped_lengths(random_order))
sorted_order = sorted(train_lengths)
similar_padding = padded_positions(train_lengths, grouped_lengths(sorted_order))
useful_tokens = sum(train_lengths)
rows = []
for name, padding in [("Fixed length 1,024", fixed_padding), ("Dynamic, seeded random batches", random_padding), ("Dynamic, similar-length batches", similar_padding)]:
    total = useful_tokens + padding
    rows.append({"Strategy": name, "Useful tokens": useful_tokens, "Padded tokens": padding, "Padding share": 100 * padding / total, "Reduction vs fixed (%)": 100 * (fixed_padding - padding) / fixed_padding})
padding_summary = pd.DataFrame(rows)
print(padding_summary.to_string(index=False, formatters={"Padding share": "{:.2f}%".format, "Reduction vs fixed (%)": "{:.2f}%".format}))
print(f"Logical batches: {math.ceil(len(train_lengths) / BATCH_SIZE)} (last batch size {len(train_lengths) % BATCH_SIZE or BATCH_SIZE})")
Path("padding-benchmarks").mkdir(exist_ok=True)
padding_summary.to_json("padding-benchmarks/token-counts.json", orient="records", indent=2)


                       Strategy  Useful tokens  Padded tokens Padding share Reduction vs fixed (%)
             Fixed length 1,024         244388         779612        76.13%                  0.00%
 Dynamic, seeded random batches         244388         183026        42.82%                 76.52%
Dynamic, similar-length batches         244388           1802         0.73%                 99.77%
Logical batches: 167 (last batch size 4)


In [7]:
# Build correctly masked causal-LM batches. Padding uses EOS as its token ID,
# so the attention mask—not token identity—decides which labels become -100.
class UnpaddedTokenDataset(Dataset):
    def __init__(self, encoded):
        self.rows = []
        for row in encoded:
            length = int(sum(row["attention_mask"]))
            self.rows.append({"input_ids": row["input_ids"][:length], "attention_mask": row["attention_mask"][:length]})
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        return self.rows[index]

class CausalPaddingCollator:
    def __init__(self, tokenizer, mode):
        self.tokenizer, self.mode = tokenizer, mode
    def __call__(self, features):
        batch = self.tokenizer.pad(
            [{"input_ids": f["input_ids"], "attention_mask": f["attention_mask"]} for f in features],
            padding="max_length" if self.mode == "fixed" else True,
            max_length=MAX_LENGTH if self.mode == "fixed" else None,
            return_tensors="pt",
        )
        labels = batch["input_ids"].clone()
        labels[batch["attention_mask"] == 0] = -100
        batch["labels"] = labels
        return batch

train_unpadded = UnpaddedTokenDataset(tokenized_dataset)
eval_unpadded = UnpaddedTokenDataset(tokenized_dataset_eval)
collator_check = CausalPaddingCollator(tokenizer, "dynamic")([
    {"input_ids": [tokenizer.eos_token_id, 17], "attention_mask": [1, 1]},
    {"input_ids": [tokenizer.eos_token_id], "attention_mask": [1]},
])
assert collator_check["labels"][0, 0].item() == tokenizer.eos_token_id
assert collator_check["labels"][1, 0].item() == tokenizer.eos_token_id
assert collator_check["labels"][1, 1].item() == -100
print("Collator check passed: attended EOS targets remain; only padded labels are ignored.")
print("Example batch tensor shapes:", {key: tuple(value.shape) for key, value in collator_check.items()})


You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Collator check passed: attended EOS targets remain; only padded labels are ignored.
Example batch tensor shapes: {'input_ids': (2, 2), 'attention_mask': (2, 2), 'labels': (2, 2)}


In [8]:
# Train from the public GPT-2 checkpoint with fixed 1,024-token padding. This
# is a fresh, matched timing run; the earlier medical-domain checkpoint is untouched.
def sync_device():
    if device == "mps":
        torch.mps.synchronize()
    elif device == "cuda":
        torch.cuda.synchronize()

def train_padding_variant(name, mode):
    from transformers import Trainer, TrainingArguments
    output_dir = f"./padding-benchmarks/{name}"
    model = AutoModelForCausalLM.from_pretrained(model_name, attn_implementation="sdpa")
    model.config.use_cache = False
    args = TrainingArguments(
        output_dir=output_dir, overwrite_output_dir=True, max_steps=12,
        per_device_train_batch_size=1, gradient_accumulation_steps=6,
        per_device_eval_batch_size=1, save_strategy="no", logging_dir=f"{output_dir}/logs",
        logging_steps=6, eval_strategy="no", report_to="none",
        seed=42, data_seed=42, dataloader_pin_memory=(device == "cuda"),
    )
    trainer = Trainer(model=model, args=args, train_dataset=train_unpadded,
                      eval_dataset=eval_unpadded, data_collator=CausalPaddingCollator(tokenizer, mode))
    sync_device()
    started = time.perf_counter()
    result = trainer.train()
    sync_device()
    elapsed = time.perf_counter() - started
    record = {"variant": name, "padding": mode, "wall_seconds": elapsed,
              "trainer_metrics": result.metrics, "global_step": trainer.state.global_step,
              "final_epoch": trainer.state.epoch, "device": device, "seed": 42,
              "train_examples": len(train_unpadded), "eval_examples": len(eval_unpadded),
              "max_steps": 12, "evaluation": "disabled for matched short timing run",
              "microbatch": 1, "gradient_accumulation_steps": 6}
    result_path = Path("padding-benchmarks/training-results.json")
    prior = json.loads(result_path.read_text()) if result_path.exists() else []
    prior.append(record)
    result_path.write_text(json.dumps(prior, indent=2, default=str) + "\n")
    del trainer, model
    import gc
    gc.collect()
    if device == "mps": torch.mps.empty_cache()
    if device == "cuda": torch.cuda.empty_cache()
    print(json.dumps(record, indent=2, default=str))
    return record


In [9]:
# Both variants start from the same public checkpoint and use identical data,
# seeds, optimizer, accumulation, and a matched 12-update training window.
fixed_run = train_padding_variant("fixed-padding", "fixed")

max_steps is given, it will override any value given in num_train_epochs


Step,Training Loss
6,3.400100
12,3.430300


{
  "variant": "fixed-padding",
  "padding": "fixed",
  "wall_seconds": 207.01193416694878,
  "trainer_metrics": {
    "train_runtime": 206.869,
    "train_samples_per_second": 0.348,
    "train_steps_per_second": 0.058,
    "total_flos": 37626052608000.0,
    "train_loss": 3.4151899019877114,
    "epoch": 0.072
  },
  "global_step": 12,
  "final_epoch": 0.072,
  "device": "mps",
  "seed": 42,
  "train_examples": 1000,
  "eval_examples": 500,
  "max_steps": 12,
  "evaluation": "disabled for matched short timing run",
  "microbatch": 1,
  "gradient_accumulation_steps": 6
}


In [9]:
# Dynamic padding shortens each one-example physical microbatch to its content.
# With microbatch=1, length grouping cannot further reduce training padding; its
# measurable benefit is the six-example logical-batch comparison above.
dynamic_run = train_padding_variant("dynamic-padding", "dynamic")

max_steps is given, it will override any value given in num_train_epochs


Step,Training Loss
6,3.412100
12,3.418800


{
  "variant": "dynamic-padding",
  "padding": "dynamic",
  "wall_seconds": 467.3461262079654,
  "trainer_metrics": {
    "train_runtime": 467.2641,
    "train_samples_per_second": 0.154,
    "train_steps_per_second": 0.026,
    "total_flos": 9128380032000.0,
    "train_loss": 3.415445327758789,
    "epoch": 0.072
  },
  "global_step": 12,
  "final_epoch": 0.072,
  "device": "mps",
  "seed": 42,
  "train_examples": 1000,
  "eval_examples": 500,
  "max_steps": 12,
  "evaluation": "disabled for matched short timing run",
  "microbatch": 1,
  "gradient_accumulation_steps": 6
}


In [10]:
# Reconcile actual measurements and token-count analysis without assuming
# dynamic padding is faster on this hardware. These are 12-update single runs.
training_runs = json.loads(Path("padding-benchmarks/training-results.json").read_text())
matched = [run for run in training_runs if run.get("max_steps") == 12]
assert {run["variant"] for run in matched} >= {"fixed-padding", "dynamic-padding"}
print(padding_summary.to_string(index=False, formatters={"Padding share": "{:.2f}%".format, "Reduction vs fixed (%)": "{:.2f}%".format}))
print("\nMatched 12-update training measurements (evaluation disabled):")
for run in matched:
    print(f"{run['variant']}: {run['wall_seconds']:.2f} s; steps={run['global_step']}; loss={run['trainer_metrics'].get('train_loss')}")
print("\nTiming is one controlled run per variant on this device; no speedup is assumed.")

                       Strategy  Useful tokens  Padded tokens Padding share Reduction vs fixed (%)
             Fixed length 1,024         244388         779612        76.13%                  0.00%
 Dynamic, seeded random batches         244388         183026        42.82%                 76.52%
Dynamic, similar-length batches         244388           1802         0.73%                 99.77%

Matched 12-update training measurements (evaluation disabled):
fixed-padding: 207.01 s; steps=12; loss=3.4151899019877114
dynamic-padding: 467.35 s; steps=12; loss=3.415445327758789

Timing is one controlled run per variant on this device; no speedup is assumed.


### What the padding comparison establishes

Fixed padding makes every row length 1,024. Dynamic padding instead pads only to the longest item in each batch; grouping similar lengths makes those batch maxima smaller. These token counts are exact for the selected 1,000 examples and seeded order. The timing comparison is a single matched run per policy on the recorded device, so it demonstrates this run’s observed cost rather than a universal speedup. With the required physical microbatch of one, dynamic padding removes padded positions from training computation, while grouping is evaluated in the six-example accounting above.
